In [0]:
from pyspark.sql import functions as F

In [0]:
def log(msg):
    print(f"[INFO] {msg}")

def error(msg):
    print(f"[ERROR] {msg}")

In [0]:
try:
    log("Reading Silver tables")

    row_count_df = spark.table("dq_project.silver.row_count_diff")
    schema_diff_df = spark.table("dq_project.silver.schema_diff")
    null_df = spark.table("dq_project.silver.null_diff")
    key_df = spark.table("dq_project.silver.key_quality")
    avg_df = spark.table("dq_project.silver.avg_diff")

    log("Silver tables loaded successfully")

except Exception as e:
    error(f"Failed to read Silver tables: {str(e)}")
    raise

[INFO] Reading Silver tables
[INFO] Silver tables loaded successfully


-  ##  Validation Report

In [0]:
try:
    log("Creating validation summary")

    row_diff = row_count_df.collect()[0]["difference"]
    schema_issue_count = schema_diff_df.count()
    null_issue_count = null_df.filter(F.col("source_nulls") != F.col("target_nulls")).count()
    key_issue_count = key_df.filter(
        (F.col("source_duplicates") > 0) | (F.col("target_duplicates") > 0)
    ).count()

    summary_data = [
        ("Row Count Check", "PASS" if row_diff == 0 else "FAIL"),
        ("Schema Check", "PASS" if schema_issue_count == 0 else "CHECK"),
        ("Null Check", "PASS" if null_issue_count == 0 else "CHECK"),
        ("Key Check", "PASS" if key_issue_count == 0 else "CHECK")
    ]

    validation_summary_df = spark.createDataFrame(
        summary_data,
        ["check_name", "status"]
    )

    display(validation_summary_df)

except Exception as e:
    error(f"Validation summary creation failed: {str(e)}")
    raise

[INFO] Creating validation summary


check_name,status
Row Count Check,FAIL
Schema Check,CHECK
Null Check,CHECK
Key Check,PASS


In [0]:
try:
    log("Saving validation_summary")

    validation_summary_df.write.mode("overwrite").saveAsTable("dq_project.gold.validation_summary")

    log("validation_summary saved successfully")

except Exception as e:
    error(f"Saving validation_summary failed: {str(e)}")
    raise

[INFO] Saving validation_summary
[INFO] validation_summary saved successfully


## - Comparison Report

- Row Count Section

In [0]:
try:
    log("Preparing row count report")

    row_report_df = row_count_df.select(
        F.lit("row_count").alias("check_type"),
        F.lit("all_rows").alias("column_name"),
        F.col("source_count").cast("string").alias("source_value"),
        F.col("target_count").cast("string").alias("target_value"),
        F.col("difference").cast("string").alias("difference_value")
    )

    display(row_report_df)

except Exception as e:
    error(f"Row report failed: {str(e)}")
    raise

[INFO] Preparing row count report


check_type,column_name,source_value,target_value,difference_value
row_count,all_rows,2500,2520,20


Schema Section

In [0]:
try:
    log("Preparing schema report")

    if schema_diff_df.count() == 0:
        schema_report_df = spark.createDataFrame(
            [("schema_check", "no_difference", "", "", "PASS")],
            ["check_type", "column_name", "source_value", "target_value", "difference_value"]
        )
    else:
        schema_report_df = schema_diff_df.select(
            F.lit("schema_check").alias("check_type"),
            F.col("column_name"),
            F.lit("").alias("source_value"),
            F.lit("").alias("target_value"),
            F.col("difference").alias("difference_value")
        )

    display(schema_report_df)

except Exception as e:
    error(f"Schema report failed: {str(e)}")
    raise

[INFO] Preparing schema report


check_type,column_name,source_value,target_value,difference_value
schema_check,preferred_channel,,,only_in_target
schema_check,risk_band,,,only_in_source


Null Section

In [0]:

try:
    log("Preparing null report")

    null_report_df = null_df.select(
        F.lit("null_check").alias("check_type"),
        F.col("column").alias("column_name"),
        F.col("source_nulls").cast("string").alias("source_value"),
        F.col("target_nulls").cast("string").alias("target_value"),
        (F.col("target_nulls") - F.col("source_nulls")).cast("string").alias("difference_value")
    )

    display(null_report_df)

except Exception as e:
    error(f"Null report failed: {str(e)}")
    raise

[INFO] Preparing null report


check_type,column_name,source_value,target_value,difference_value
null_check,loyalty_score,0,0,0
null_check,dataset_type,0,0,0
null_check,ingestion_ts,0,0,0
null_check,first_name,0,0,0
null_check,tenure_months,0,0,0
null_check,city,65,158,93
null_check,last_name,0,0,0
null_check,gender,0,0,0
null_check,is_active,0,0,0
null_check,customer_id,0,0,0


- Average Section

In [0]:
try:
    log("Preparing average report")

    avg_report_df = avg_df.select(
        F.lit("average_check").alias("check_type"),
        F.col("column").alias("column_name"),
        F.col("source_avg").cast("string").alias("source_value"),
        F.col("target_avg").cast("string").alias("target_value"),
        F.col("avg_difference").cast("string").alias("difference_value")
    )

    display(avg_report_df)

except Exception as e:
    error(f"Average report failed: {str(e)}")
    raise

[INFO] Preparing average report


check_type,column_name,source_value,target_value,difference_value
average_check,purchase_count,13.9792,15.638095238095238,1.658895238095237
average_check,loyalty_score,73.86568,75.33083333333336,1.4651533333333617
average_check,annual_income,845893.8056910569,904966.2390143737,59072.43332331686
average_check,credit_score,708.3317191283293,696.5274193548387,-11.804299773490584
average_check,tenure_months,48.8756,48.801984126984124,-0.07361587301587491
average_check,is_active,0.788,0.7880952380952381,9.523809523803717E-5
average_check,customer_id,101250.5,101260.5,10.0
average_check,age,36.4576,36.46626984126984,0.008669841269842493


-  Key Section

In [0]:
try:
    log("Preparing key report")

    key_report_df = key_df.select(
        F.lit("key_check").alias("check_type"),
        F.col("key_column").alias("column_name"),
        F.col("source_duplicates").cast("string").alias("source_value"),
        F.col("target_duplicates").cast("string").alias("target_value"),
        F.lit("").alias("difference_value")
    )

    display(key_report_df)

except Exception as e:
    error(f"Key report failed: {str(e)}")
    raise

[INFO] Preparing key report


check_type,column_name,source_value,target_value,difference_value
key_check,customer_id,0,0,


- Combine All Reports

In [0]:
try:
    log("Combining all reports")

    comparison_report_df = (
        row_report_df
        .unionByName(schema_report_df)
        .unionByName(null_report_df)
        .unionByName(avg_report_df)
        .unionByName(key_report_df)
    )

    display(comparison_report_df)

except Exception as e:
    error(f"Combining reports failed: {str(e)}")
    raise

[INFO] Combining all reports


check_type,column_name,source_value,target_value,difference_value
row_count,all_rows,2500,2520,20
schema_check,preferred_channel,,,only_in_target
schema_check,risk_band,,,only_in_source
null_check,loyalty_score,0,0,0
null_check,dataset_type,0,0,0
null_check,ingestion_ts,0,0,0
null_check,first_name,0,0,0
null_check,tenure_months,0,0,0
null_check,city,65,158,93
null_check,last_name,0,0,0



- Saving Comparison Report

In [0]:
try:
    log("Saving comparison_report")

    comparison_report_df.write.mode("overwrite").saveAsTable("dq_project.gold.comparison_report")

    log("comparison_report saved successfully")

except Exception as e:
    error(f"Saving comparison_report failed: {str(e)}")
    raise

[INFO] Saving comparison_report
[INFO] comparison_report saved successfully


## - Insights From the Comparison

- Customer Count Insight

In [0]:
try:
    log("Creating customer count insight")

    row_data = row_count_df.collect()[0]

    if row_data["source_count"] > row_data["target_count"]:
        msg = "Source dataset has more customers than target dataset"
    elif row_data["source_count"] < row_data["target_count"]:
        msg = "Target dataset has more customers than source dataset"
    else:
        msg = "Both datasets have equal number of customers"

    print(msg)

except Exception as e:
    error(f"Customer count insight failed: {str(e)}")
    raise

[INFO] Creating customer count insight
Target dataset has more customers than source dataset


-  Null Comparison Insight

In [0]:
try:
    log("Creating null insight")

    top_null = null_df.orderBy(
        F.desc(F.abs(F.col("source_nulls") - F.col("target_nulls")))
    ).limit(1).collect()[0]

    if top_null["source_nulls"] > top_null["target_nulls"]:
        msg = f"Source dataset has more missing values in column '{top_null['column']}'"
    else:
        msg = f"Target dataset has more missing values in column '{top_null['column']}'"

    print(msg)

except Exception as e:
    error(f"Null insight failed: {str(e)}")
    raise

[INFO] Creating null insight
Target dataset has more missing values in column 'city'


- Average Comparison Insight

In [0]:
try:
    log("Creating average insight")

    top_avg = avg_df.orderBy(
        F.desc(F.abs(F.col("avg_difference")))
    ).limit(1).collect()[0]

    if top_avg["avg_difference"] > 0:
        msg = f"Target dataset has higher average value in column '{top_avg['column']}'"
    else:
        msg = f"Source dataset has higher average value in column '{top_avg['column']}'"

    print(msg)

except Exception as e:
    error(f"Average insight failed: {str(e)}")
    raise

[INFO] Creating average insight
Target dataset has higher average value in column 'annual_income'


- Key Quality Comparison Insight

In [0]:
try:
    log("Creating key quality insight")

    key_row = key_df.collect()[0]

    if key_row["source_duplicates"] > key_row["target_duplicates"]:
        msg = "Source dataset has more duplicate customer IDs"
    elif key_row["source_duplicates"] < key_row["target_duplicates"]:
        msg = "Target dataset has more duplicate customer IDs"
    else:
        msg = "Both datasets have same level of duplicate customer IDs"

    print(msg)

except Exception as e:
    error(f"Key insight failed: {str(e)}")
    raise

[INFO] Creating key quality insight
Both datasets have same level of duplicate customer IDs


- Combining all bussiness Insights

In [0]:
try:
    log("Creating final business insights table")

    row_data = row_count_df.collect()[0]
    key_row = key_df.collect()[0]
    top_null = null_df.orderBy(F.desc(F.abs(F.col("source_nulls") - F.col("target_nulls")))).limit(1).collect()[0]
    top_avg = avg_df.orderBy(F.desc(F.abs(F.col("avg_difference")))).limit(1).collect()[0]

    # Row insight
    if row_data["source_count"] > row_data["target_count"]:
        row_msg = "Source dataset has more customers"
    elif row_data["source_count"] < row_data["target_count"]:
        row_msg = "Target dataset has more customers"
    else:
        row_msg = "Both datasets have equal customers"

    # Null insight
    if top_null["source_nulls"] > top_null["target_nulls"]:
        null_msg = f"More missing values in source for column '{top_null['column']}'"
    else:
        null_msg = f"More missing values in target for column '{top_null['column']}'"

    # Avg insight
    if top_avg["avg_difference"] > 0:
        avg_msg = f"Target has higher average in '{top_avg['column']}'"
    else:
        avg_msg = f"Source has higher average in '{top_avg['column']}'"

    # Key insight
    if key_row["source_duplicates"] > key_row["target_duplicates"]:
        key_msg = "More duplicate IDs in source"
    elif key_row["source_duplicates"] < key_row["target_duplicates"]:
        key_msg = "More duplicate IDs in target"
    else:
        key_msg = "Duplicate levels are similar"

    insights_data = [
        ("Customer Count", row_msg),
        ("Missing Data", null_msg),
        ("Average Comparison", avg_msg),
        ("Key Quality", key_msg)
    ]

    insights_df = spark.createDataFrame(
        insights_data,
        ["insight_type", "insight"]
    )

    display(insights_df)

except Exception as e:
    error(f"Insights creation failed: {str(e)}")
    raise

[INFO] Creating final business insights table


insight_type,insight
Customer Count,Target dataset has more customers
Missing Data,More missing values in target for column 'city'
Average Comparison,Target has higher average in 'annual_income'
Key Quality,Duplicate levels are similar


- Saving Insights as tabel

In [0]:
try:
    log("Saving business insights")

    insights_df.write.mode("overwrite").saveAsTable("dq_project.gold.interesting_insights")

    log("Insights saved successfully")

except Exception as e:
    error(f"Saving insights failed: {str(e)}")
    raise

[INFO] Saving business insights
[INFO] Insights saved successfully


- Displaying  Vadilation_summary_report

In [0]:
display(spark.table("dq_project.gold.validation_summary"))


check_name,status
Row Count Check,FAIL
Schema Check,CHECK
Null Check,CHECK
Key Check,PASS


- Displaying Comparison_Report 

In [0]:
display(spark.table("dq_project.gold.comparison_report"))


check_type,column_name,source_value,target_value,difference_value
average_check,annual_income,845893.8056910569,904966.2390143737,59072.43332331686
average_check,credit_score,708.3317191283293,696.5274193548387,-11.804299773490584
average_check,tenure_months,48.8756,48.801984126984124,-0.07361587301587491
average_check,is_active,0.788,0.7880952380952381,9.523809523803717E-5
average_check,purchase_count,13.9792,15.638095238095238,1.658895238095237
average_check,loyalty_score,73.86568,75.33083333333336,1.4651533333333617
average_check,customer_id,101250.5,101260.5,10.0
average_check,age,36.4576,36.46626984126984,0.008669841269842493
null_check,loyalty_score,0,0,0
null_check,dataset_type,0,0,0


- Displaying Compared Bussiness Insights

In [0]:
display(spark.table("dq_project.gold.interesting_insights"))

insight_type,insight
Average Comparison,Target has higher average in 'annual_income'
Missing Data,More missing values in target for column 'city'
Customer Count,Target dataset has more customers
Key Quality,Duplicate levels are similar


## - Extracting  chartable parts from comparison_report

- Row Count Comparison Chart Visualization

In [0]:
from pyspark.sql import functions as F

row_chart_df = (
    spark.table("dq_project.gold.comparison_report")
    .filter(F.col("check_type") == "row_count")
    .selectExpr(
        "stack(2, 'Source', cast(source_value as double), 'Target', cast(target_value as double)) as (dataset, value)"
    )
)

display(row_chart_df)

dataset,value
Source,2500.0
Target,2520.0


Databricks visualization. Run in Databricks to view.

- Null Diffrence Comparison Visualization 

In [0]:
null_chart_df = (
    spark.table("dq_project.gold.comparison_report")
    .filter(F.col("check_type") == "null_check")
    .select(
        F.col("column_name"),
        F.col("difference_value").cast("double").alias("null_difference")
    )
)

display(null_chart_df)

column_name,null_difference
loyalty_score,0.0
dataset_type,0.0
ingestion_ts,0.0
first_name,0.0
tenure_months,0.0
city,93.0
last_name,0.0
gender,0.0
is_active,0.0
customer_id,0.0


Databricks visualization. Run in Databricks to view.

- Average Diffrence Comparison Visualization

In [0]:
avg_chart_df = (
    spark.table("dq_project.gold.comparison_report")
    .filter(F.col("check_type") == "average_check")
    .select(
        F.col("column_name"),
        F.col("source_value").cast("double").alias("source_avg"),
        F.col("target_value").cast("double").alias("target_avg"),
        F.col("difference_value").cast("double").alias("avg_difference")
    )
)

display(avg_chart_df)

column_name,source_avg,target_avg,avg_difference
annual_income,845893.8056910569,904966.2390143737,59072.43332331686
credit_score,708.3317191283293,696.5274193548387,-11.804299773490584
tenure_months,48.8756,48.801984126984124,-0.07361587301587491
is_active,0.788,0.7880952380952381,9.523809523803717E-5
purchase_count,13.9792,15.638095238095238,1.658895238095237
loyalty_score,73.86568,75.33083333333336,1.4651533333333617
customer_id,101250.5,101260.5,10.0
age,36.4576,36.46626984126984,0.008669841269842493


Databricks visualization. Run in Databricks to view.